In [57]:
import pandas as pd
import skimage as ski
import mahotas
import numpy as np
import os
from pathlib import Path

In [58]:
PATH = Path("../data/metadata.csv")
df = pd.read_csv(PATH)
df


,Unnamed: 0.1,Unnamed: 0,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,...,hurt,changed,bleed,elevation,img_id,biopsed,group_id,JM_Hair,JM_Pen,JM_Pus
0,16,17,PAT_1783,3414,NaN,NaN,NaN,NaN,75,NaN,...,FALSE,FALSE,FALSE,True,PAT_1783_3414_120.png,False,J,1.0,NaN,NaN
1,27,28,PAT_302,651,True,False,GERMANY,GERMANY,64,True,...,TRUE,FALSE,TRUE,True,PAT_302_651_529.png,True,J,1.0,1.0,NaN
2,77,79,PAT_1094,381,NaN,NaN,NaN,NaN,41,NaN,...,FALSE,FALSE,FALSE,False,PAT_1094_381_85.png,False,J,NaN,NaN,NaN
3,78,80,PAT_645,4043,False,False,GERMANY,GERMANY,57,True,...,FALSE,FALSE,FALSE,False,PAT_645_4043_374.png,False,J,NaN,NaN,NaN
4,112,115,PAT_549,1043,False,False,BRAZIL,BRAZIL,85,False,...,FALSE,FALSE,FALSE,True,PAT_549_1043_676.png,True,J,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,2027,2183,PAT_682,1292,False,False,POMERANIA,POMERANIA,75,False,...,FALSE,FALSE,TRUE,True,PAT_682_1292_642.png,True,J,NaN,NaN,NaN
113,2043,2205,PAT_223,1067,False,False,GERMANY,BRAZIL,73,False,...,FALSE,FALSE,FALSE,True,PAT_223_1067_569.png,False,J,NaN,1.0,NaN
114,2057,2227,PAT_726,1371,False,False,POMERANIA,POMERANIA,56,False,...,FALSE,FALSE,FALSE,True,PAT_726_1371_397.png,True,J,NaN,NaN,NaN
115,2062,2237,PAT_30,41,False,False,GERMANY,GERMANY,57,False,...,FALSE,UNK,FALSE,False,PAT_30_41_815.png,True,J,0.0,0.0,0.0


In [59]:
# Filter away maskless images
mask_dir = Path("../data/masks")

imgID = df["img_id"].to_list()

maskExists = []

for i in imgID:
    base_name = Path(i).name
    name = Path(base_name).stem 
    ext = Path(base_name).suffix  
    
    mask_path = mask_dir / f"{name}_mask{ext}"
    maskExists.append(mask_path.exists())

In [60]:
df = df[maskExists]
df

,Unnamed: 0.1,Unnamed: 0,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,...,hurt,changed,bleed,elevation,img_id,biopsed,group_id,JM_Hair,JM_Pen,JM_Pus
0,16,17,PAT_1783,3414,NaN,NaN,NaN,NaN,75,NaN,...,FALSE,FALSE,FALSE,True,PAT_1783_3414_120.png,False,J,1.0,NaN,NaN
1,27,28,PAT_302,651,True,False,GERMANY,GERMANY,64,True,...,TRUE,FALSE,TRUE,True,PAT_302_651_529.png,True,J,1.0,1.0,NaN
2,77,79,PAT_1094,381,NaN,NaN,NaN,NaN,41,NaN,...,FALSE,FALSE,FALSE,False,PAT_1094_381_85.png,False,J,NaN,NaN,NaN
3,78,80,PAT_645,4043,False,False,GERMANY,GERMANY,57,True,...,FALSE,FALSE,FALSE,False,PAT_645_4043_374.png,False,J,NaN,NaN,NaN
4,112,115,PAT_549,1043,False,False,BRAZIL,BRAZIL,85,False,...,FALSE,FALSE,FALSE,True,PAT_549_1043_676.png,True,J,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,2027,2183,PAT_682,1292,False,False,POMERANIA,POMERANIA,75,False,...,FALSE,FALSE,TRUE,True,PAT_682_1292_642.png,True,J,NaN,NaN,NaN
113,2043,2205,PAT_223,1067,False,False,GERMANY,BRAZIL,73,False,...,FALSE,FALSE,FALSE,True,PAT_223_1067_569.png,False,J,NaN,1.0,NaN
114,2057,2227,PAT_726,1371,False,False,POMERANIA,POMERANIA,56,False,...,FALSE,FALSE,FALSE,True,PAT_726_1371_397.png,True,J,NaN,NaN,NaN
115,2062,2237,PAT_30,41,False,False,GERMANY,GERMANY,57,False,...,FALSE,UNK,FALSE,False,PAT_30_41_815.png,True,J,0.0,0.0,0.0


In [ ]:
img_dir = Path("../data/imgs")
mask_dir = Path("../data/masks")

imgID = df["img_id"].to_list()
diagnosis = df["diagnostic"].to_list()
compactnesses = []
darkCos = []

for i in range(len(imgID)):
    img_path = img_dir / imgID[i]
    mask_path = mask_dir / f"{img_path.stem}_mask{img_path.suffix}"

    img = ski.io.imread(img_path)
    img = ski.transform.resize(img, (255, 255))

    mask = ski.io.imread(mask_path)
    mask = ski.transform.resize(mask, (255, 255), preserve_range=True)

    # mask to boolean
    if mask.ndim == 3:
        mask = ski.color.rgb2gray(mask) > 0.5
    else:
        mask = mask > 0.5

    #compactness
    area = mask.sum()
    perimeter = area - ski.morphology.erosion(mask,ski.morphology.disk(3)).sum()  # erosion can be refined to be even finer
    compactness = perimeter**2 / (area * 12)
    compactnesses.append(compactness)

    #darkness coefficient
    # takes how many pixels are light vs dark
    darkCo = (img[mask][:, :3].sum(axis=1)/3 > 0.5).sum() / img[mask].shape[0]
    darkCos.append(darkCo)

# Add to dataframe
df["compactness"] = compactnesses
df["dark"] = darkCos

np.float64(160.99667787882564)

In [63]:
#save as feature extracted CSV
PATH = Path("../data/metadataImgFeat.csv")
df.to_csv(PATH)